1. Preprocessing Text in NLP ( Tokenization, Filtration, Script validation, Stop Word Removal, Stemming).

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

# Download required resources
nltk.download('punkt_tab') # Tokenizer
nltk.download('stopwords') # Stop words

# Input text
text = "NLP is a fascinating field of study! It involves tasks like tokenization, filtration, and stemming."

### 1. Tokenization
# Split the text into words
tokens = word_tokenize(text)
print("Tokenized Words:", tokens)

### 2. Filtration (e.g., removing punctuation and non-alphanumeric characters)
filtered_tokens = [word for word in tokens if word.isalnum()]
print("Filtered Tokens:", filtered_tokens)

### 3. Script Validation (e.g., ensuring all tokens are in English alphabet)
validated_tokens = [word for word in filtered_tokens if re.match(r'^[A-Za-z]+$', word)]
print("Script Validated Tokens:", validated_tokens)

### 4. Stop Word Removal
# Define English stop words
stop_words = set(stopwords.words('english'))
tokens_without_stopwords = [word for word in validated_tokens if word.lower() not in stop_words]
print("Tokens without Stop Words:", tokens_without_stopwords)

### 5. Stemming
# Initialize Porter Stemmer
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(word) for word in tokens_without_stopwords]
print("Stemmed Tokens:", stemmed_tokens)

# Final preprocessed text
preprocessed_text = " ".join(stemmed_tokens)
print("\nFinal Preprocessed Text:", preprocessed_text)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Tokenized Words: ['NLP', 'is', 'a', 'fascinating', 'field', 'of', 'study', '!', 'It', 'involves', 'tasks', 'like', 'tokenization', ',', 'filtration', ',', 'and', 'stemming', '.']
Filtered Tokens: ['NLP', 'is', 'a', 'fascinating', 'field', 'of', 'study', 'It', 'involves', 'tasks', 'like', 'tokenization', 'filtration', 'and', 'stemming']
Script Validated Tokens: ['NLP', 'is', 'a', 'fascinating', 'field', 'of', 'study', 'It', 'involves', 'tasks', 'like', 'tokenization', 'filtration', 'and', 'stemming']
Tokens without Stop Words: ['NLP', 'fascinating', 'field', 'study', 'involves', 'tasks', 'like', 'tokenization', 'filtration', 'stemming']
Stemmed Tokens: ['nlp', 'fascin', 'field', 'studi', 'involv', 'task', 'like', 'token', 'filtrat', 'stem']

Final Preprocessed Text: nlp fascin field studi involv task like token filtrat stem


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


2. N-gram modeling to analyze and establish probability distribution and exploring unigrams, bigrams and trigrams.

In [ ]:
import nltk
from nltk.util import ngrams
from collections import Counter
import math

# Sample sentences
sentences = [
    "I love natural language processing",
    "I love programming",
    "Natural language processing is fascinating",
    "Programming is fun and rewarding"
]

# Preprocessing the corpus: Tokenize and Lowercase each sentence
tokenized_sentences = [nltk.word_tokenize(sentence.lower()) for sentence in sentences]

# Function to generate n-grams from tokenized sentences
def generate_ngrams(tokenized_sentences, n):
    ngrams_list = []
    for sentence in tokenized_sentences:
        ngrams_list.extend(ngrams(sentence, n))
    return ngrams_list

# Function to calculate probabilities
def calculate_ngram_probabilities(ngrams_list, n_minus_1_grams=None):
    ngram_counts = Counter(ngrams_list)
    probabilities = {}

    if n_minus_1_grams:  # For bigrams, trigrams, etc.
        n_minus_1_counts = Counter(n_minus_1_grams)
        for ngram in ngram_counts:
            prefix = ngram[:-1]
            probabilities[ngram] = ngram_counts[ngram] / n_minus_1_counts[prefix]
    else:  # For unigrams
        total_count = sum(ngram_counts.values())
        for ngram in ngram_counts:
            probabilities[ngram] = ngram_counts[ngram] / total_count

    return probabilities

# Generate unigrams, bigrams, and trigrams
unigrams = generate_ngrams(tokenized_sentences, 1)
bigrams = generate_ngrams(tokenized_sentences, 2)
trigrams = generate_ngrams(tokenized_sentences, 3)

# Calculate probabilities
unigram_probabilities = calculate_ngram_probabilities(unigrams)
bigram_probabilities = calculate_ngram_probabilities(bigrams, unigrams)
trigram_probabilities = calculate_ngram_probabilities(trigrams, bigrams)

# Display probabilities
print("Unigram Probabilities:")
for unigram, prob in unigram_probabilities.items():
    print(f"{unigram}: {prob:.4f}")

print("\nBigram Probabilities:")
for bigram, prob in bigram_probabilities.items():
    print(f"{bigram}: {prob:.4f}")

print("\nTrigram Probabilities:")
for trigram, prob in trigram_probabilities.items():
    print(f"{trigram}: {prob:.4f}")

# Example usage: Calculating sentence probabilities
def calculate_sentence_probability(sentence, n, probabilities):
    tokens = nltk.word_tokenize(sentence.lower())
    ngrams_sentence = list(
        ngrams(
            tokens, n,
            pad_left=True, pad_right=True,
            left_pad_symbol="<s>", right_pad_symbol="</s>"
        )
    )
    probability = 1.0
    for ngram in ngrams_sentence:
        prob = probabilities.get(ngram, 0)  # Default to 0 for unseen n-grams
        if prob == 0:
            probability *= 0.0001  # Smoothing for unseen n-grams
        else:
            probability *= prob
    return probability

# Test sentences
test_sentence = "I love programming"
print("Sentence Probability (Unigram):",
      calculate_sentence_probability(test_sentence, 1, unigram_probabilities))
print("Sentence Probability (Bigram):",
      calculate_sentence_probability(test_sentence, 2, bigram_probabilities))
print("Sentence Probability (Trigram):",
      calculate_sentence_probability(test_sentence, 3, trigram_probabilities))


Unigram Probabilities:
('i',): 0.1111
('love',): 0.1111
('natural',): 0.1111
('language',): 0.1111
('processing',): 0.1111
('programming',): 0.1111
('is',): 0.1111
('fascinating',): 0.0556
('fun',): 0.0556
('and',): 0.0556
('rewarding',): 0.0556

Bigram Probabilities:
('i', 'love'): 1.0000
('love', 'natural'): 0.5000
('natural', 'language'): 1.0000
('language', 'processing'): 1.0000
('love', 'programming'): 0.5000
('processing', 'is'): 0.5000
('is', 'fascinating'): 0.5000
('programming', 'is'): 0.5000
('is', 'fun'): 0.5000
('fun', 'and'): 1.0000
('and', 'rewarding'): 1.0000

Trigram Probabilities:
('i', 'love', 'natural'): 0.5000
('love', 'natural', 'language'): 1.0000
('natural', 'language', 'processing'): 1.0000
('i', 'love', 'programming'): 0.5000
('language', 'processing', 'is'): 0.5000
('processing', 'is', 'fascinating'): 1.0000
('programming', 'is', 'fun'): 1.0000
('is', 'fun', 'and'): 1.0000
('fun', 'and', 'rewarding'): 1.0000
Sentence Probability (Unigram): 0.001371742112482853

3. Minimum Edit Distance (MED) algorithm, application in string comparison and the minimum number of edit operations required to transform one string into another.

In [ ]:
def min_edit_distance(s1, s2):
    n = len(s1)
    m = len(s2)

    # Create a (m+1)x(n+1) matrix
    dp = [[0 for _ in range(n+1)] for _ in range(m+1)]

    # Initialize base cases (transforming a string to an empty string)
    for i in range(m+1):
        dp[i][0] = i
    for j in range(n+1):
        dp[0][j] = j

    # Fill the matrix using the recurrence relation
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[j-1] == s2[i-1]:
                dp[i][j] = dp[i-1][j-1]  # No operation needed if characters are the same
            else:
                dp[i][j] = min(dp[i][j-1], dp[i-1][j], dp[i-1][j-1]) + 1  # Minimum of substitution, deletion, or insertion

    # The value in dp[m][n] is the minimum edit distance
    return dp[m][n]

# Test the algorithm on different types of variations
test_cases = [
    ("kitten", "sitting"),  # Substitution, insertion
    ("flaw", "lawn"),  # Substitution, insertion
    ("intention", "execution"),  # Substitution, insertion, deletion
    ("sunday", "saturday"),  # Substitution, insertion, deletion
    ("hello", "helo"),  # Deletion
    ("abc", "def")  # Substitution
]

for s1, s2 in test_cases:
    distance = min_edit_distance(s1, s2)
    print(f"Minimum Edit Distance between '{s1}' and '{s2}': {distance}")


Minimum Edit Distance between 'kitten' and 'sitting': 3
Minimum Edit Distance between 'flaw' and 'lawn': 2
Minimum Edit Distance between 'intention' and 'execution': 5
Minimum Edit Distance between 'sunday' and 'saturday': 3
Minimum Edit Distance between 'hello' and 'helo': 1
Minimum Edit Distance between 'abc' and 'def': 3


 4. Program to implement top-down and bottom-up parser using appropriate context free

In [ ]:
import nltk
from nltk import CFG
from nltk.parse import TopDownChartParser, BottomUpChartParser

# Define a corrected Context-Free Grammar (CFG)
grammar = CFG.fromstring("""
  S -> NP VP
  NP -> Det N | Det Adj N | N
  VP -> V NP | V
  Det -> 'a' | 'the'
  N -> 'dog' | 'cat' | 'ball' | 'park'
  Adj -> 'big' | 'small'
  V -> 'chased' | 'saw'
""")

# Define the input sentence
sentence = "the big dog saw a ball".split()

# Top-Down Parsing using TopDownChartParser
print("=== Top-Down Parsing ===")
top_down_parser = TopDownChartParser(grammar)

# Printing parsing steps for Top-Down
for step, tree in enumerate(top_down_parser.parse(sentence), start=1):
    print(f"Step {step}: {tree}")
    tree.pretty_print()

# Bottom-Up Parsing using BottomUpChartParser
print("\n=== Bottom-Up Parsing ===")
bottom_up_parser = BottomUpChartParser(grammar)

# Printing parsing steps for Bottom-Up
for step, tree in enumerate(bottom_up_parser.parse(sentence), start=1):
    print(f"Step {step}: {tree}")
    tree.pretty_print()

=== Top-Down Parsing ===
Step 1: (S
  (NP (Det the) (Adj big) (N dog))
  (VP (V saw) (NP (Det a) (N ball))))
         S                  
      ___|_______            
     |           VP         
     |        ___|___        
     NP      |       NP     
  ___|___    |    ___|___    
Det Adj  N   V  Det      N  
 |   |   |   |   |       |   
the big dog saw  a      ball


=== Bottom-Up Parsing ===
Step 1: (S
  (NP (Det the) (Adj big) (N dog))
  (VP (V saw) (NP (Det a) (N ball))))
         S                  
      ___|_______            
     |           VP         
     |        ___|___        
     NP      |       NP     
  ___|___    |    ___|___    
Det Adj  N   V  Det      N  
 |   |   |   |   |       |   
the big dog saw  a      ball



5.

In [ ]:
from collections import defaultdict
import math

# Training data
reviews = [
    ("fun, couple, love, love", "comedy"),
    ("fast, furious, shoot", "action"),
    ("couple, fly, fast, fun, fun", "comedy"),
    ("furious, shoot, shoot, fun", "action"),
    ("fly, fast, shoot, love", "action"),
]

# Test document
D = "fast, couple, shoot, fly".split(", ")

# Preprocess the data
def preprocess(data):
    tokens = []
    for doc, label in data:
        tokens.append((doc.split(", "), label))
    return tokens

data = preprocess(reviews)

# Vocabulary and counts
vocabulary = set()
class_word_counts = defaultdict(lambda: defaultdict(int))
class_counts = defaultdict(int)

for words, label in data:
    class_counts[label] += 1
    for word in words:
        class_word_counts[label][word] += 1
        vocabulary.add(word)

# Total vocabulary size
vocab_size = len(vocabulary)

# Total words in each class
total_words_in_class = {
    label: sum(class_word_counts[label].values())
    for label in class_counts
}

# Compute priors
total_docs = sum(class_counts.values())
priors = {
    label: class_counts[label] / total_docs
    for label in class_counts
}

# Likelihood with add-1 smoothing
def compute_likelihood(word, label):
    return (class_word_counts[label][word] + 1) / (
        total_words_in_class[label] + vocab_size
    )

# Compute posterior probabilities
posteriors = {}

for label in class_counts:
    # Start with log prior
    posteriors[label] = math.log(priors[label])

    # Add log likelihoods
    for word in D:
        posteriors[label] += math.log(compute_likelihood(word, label))

# Determine most likely class
predicted_class = max(posteriors, key=posteriors.get)

# Output results
print("Posteriors:")
for label, posterior in posteriors.items():
    print(f"{label}: {posterior}")

print(f"\nThe most likely class for the document '{' '.join(D)}' is: {predicted_class}")

Posteriors:
comedy: -9.52173897104528
action: -8.671115273688494

The most likely class for the document 'fast couple shoot fly' is: action


6.

In [ ]:
import nltk

from nltk.corpus import brown, inaugural, reuters, udhr
from nltk.probability import ConditionalFreqDist, FreqDist
from nltk.tag import UnigramTagger, RegexpTagger
from nltk.tokenize import word_tokenize
from nltk.corpus.reader.plaintext import PlaintextCorpusReader
from nltk.corpus import stopwords

# Download resources
nltk.download("brown", quiet=True)
nltk.download("inaugural", quiet=True)
nltk.download("reuters", quiet=True)
nltk.download("udhr", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("universal_tagset", quiet=True)
nltk.download("stopwords", quiet=True)

# 1. Study of Various Corpora
def study_corpora():
    print("Brown Corpus Categories:", brown.categories())
    print("First 10 words from Inaugural Corpus:", inaugural.words()[:10])
    print("Reuters Categories:", reuters.categories())
    print("UDHR Languages:", udhr.fileids())

study_corpora()

# 3. Conditional Frequency Distribution
def conditional_frequency_distribution():
    cfd = ConditionalFreqDist(
        (genre, word.lower())
        for genre in brown.categories()
        for word in brown.words(categories=genre)
    )

    print("\nTop words in 'news':")
    print(cfd["news"].most_common(10))k

conditional_frequency_distribution()

# 4. Tagged Corpora
def tagged_corpora_analysis():
    print("\nTagged Sentence:", brown.tagged_sents()[:1])
    print("Tagged Words:", brown.tagged_words()[:10])

tagged_corpora_analysis()

# 5. Most Frequent Nouns
def most_frequent_nouns():
    tagged_words = brown.tagged_words(tagset="universal")
    nouns = [word.lower() for word, tag in tagged_words if tag == "NOUN"]
    freq_dist = FreqDist(nouns)

    print("\nMost Frequent Nouns:", freq_dist.most_common(10))

most_frequent_nouns()

# 6. Dictionary Mapping
def map_words_to_properties():
    word_to_property = {
        "fast": "speed",
        "strong": "power",
        "bright": "intelligence"
    }

    print("\nWord Mapping:")
    for word, prop in word_to_property.items():
        print(f"{word} -> {prop}")

map_words_to_properties()

# 7. Taggers
def study_taggers():

    patterns = [
        (r'.*ing$', 'VERB'),
        (r'.*ed$', 'VERB'),
        (r'.*es$', 'VERB'),
        (r'.*s$', 'NOUN'),
        (r'.*', 'NOUN')
    ]

    regexp_tagger = RegexpTagger(patterns)

    # Add backoff
    unigram_tagger = UnigramTagger(
        brown.tagged_sents(categories='news')[:500],
        backoff=regexp_tagger
    )

    print("\nRule-Based:", regexp_tagger.tag(["running", "quickly", "table"]))
    print("Unigram:", unigram_tagger.tag(word_tokenize("The cat is on the mat.")))

study_taggers()

# 8. Word Segmentation (Improved)
def find_different_words(text, corpus_words):
    words = []
    text = text.lower()
    start = 0

    while start < len(text):
        found = False

        # Try longest match first
        for end in range(len(text), start, -1):
            word = text[start:end]
            if word in corpus_words:
                words.append(word)
                start = end
                found = True
                break

        if not found:
            start += 1

    return words

def calculate_score(words):
    return sum(len(word) for word in words)

plain_text = "thisisaplainsentence"

# FIX: lowercase corpus
corpus = set(word.lower() for word in brown.words())

found_words = find_different_words(plain_text, corpus)
score = calculate_score(found_words)

print("\nFound Words:", found_words)
print("Score:", score)

Brown Corpus Categories: ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']
First 10 words from Inaugural Corpus: ['Fellow', '-', 'Citizens', 'of', 'the', 'Senate', 'and', 'of', 'the', 'House']
Reuters Categories: ['acq', 'alum', 'barley', 'bop', 'carcass', 'castor-oil', 'cocoa', 'coconut', 'coconut-oil', 'coffee', 'copper', 'copra-cake', 'corn', 'cotton', 'cotton-oil', 'cpi', 'cpu', 'crude', 'dfl', 'dlr', 'dmk', 'earn', 'fuel', 'gas', 'gnp', 'gold', 'grain', 'groundnut', 'groundnut-oil', 'heat', 'hog', 'housing', 'income', 'instal-debt', 'interest', 'ipi', 'iron-steel', 'jet', 'jobs', 'l-cattle', 'lead', 'lei', 'lin-oil', 'livestock', 'lumber', 'meal-feed', 'money-fx', 'money-supply', 'naphtha', 'nat-gas', 'nickel', 'nkr', 'nzdlr', 'oat', 'oilseed', 'orange', 'palladium', 'palm-oil', 'palmkernel', 'pet-chem', 'platinum', 'potato', 'propane', 'rand', 'rape-o

7.

In [ ]:
import nltk
from nltk.corpus import wordnet

# Download WordNet if not already downloaded
nltk.download("wordnet")

def find_synonyms_antonyms(word):
    synonyms = set()
    antonyms = set()

    for synset in wordnet.synsets(word):
        # Add synonyms
        for lemma in synset.lemmas():
            synonyms.add(lemma.name())

            # Add antonyms
            if lemma.antonyms():
                for antonym in lemma.antonyms():
                    antonyms.add(antonym.name())

    return synonyms, antonyms

# Word to analyze
word = "active"

# Get synonyms and antonyms
synonyms, antonyms = find_synonyms_antonyms(word)

# Display results
print(f"Synonyms of '{word}': {', '.join(synonyms)}")
print(f"Antonyms of '{word}': {', '.join(antonyms)}")

[nltk_data] Downloading package wordnet to /root/nltk_data...


Synonyms of 'active': combat-ready, alive, active, active_agent, active_voice, fighting, dynamic, participating
Antonyms of 'active': passive_voice, quiet, stative, dormant, passive, extinct, inactive


8.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Define model and tokenizer
source_lang = "fr"   # French
target_lang = "en"   # English

model_name = f"Helsinki-NLP/opus-mt-{source_lang}-{target_lang}"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Function for translation
def translate(texts, tokenizer, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    outputs = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]

# Example parallel corpus
parallel_corpus = [
    ("Bonjour", "Hello"),
    ("Merci beaucoup", "Thank you very much"),
    ("Comment ça va ?", "How are you?"),
    ("Je suis étudiant.", "I am a student.")
]

# Back-translation (simple reuse here for demo)
source_texts = [pair[0] for pair in parallel_corpus]
back_translations = translate(source_texts, tokenizer, model)

# Augmented corpus
augmented_corpus = parallel_corpus + list(zip(back_translations, source_texts))

# Prepare training data (example format)
train_source = [pair[0] for pair in augmented_corpus]
train_target = [pair[1] for pair in augmented_corpus]

# Evaluation
test_sentences = ["Merci beaucoup", "Bonjour", "Je suis content."]
translations = translate(test_sentences, tokenizer, model)

print("\nTranslations:")
for src, pred in zip(test_sentences, translations):
    print(f"{src} -> {pred}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


Translations:
Merci beaucoup -> Thank you so much.
Bonjour -> Hello.
Je suis content. -> I'm glad.
